# 46. 선례 라이브러리 확장 (본선 준비)

## 목적
커버된 규칙 중 아직 실제 근거(승인약물/정량데이터)가 없는 것들을,
valid set 빈도가 높은 순으로 ChEMBL/GtoPdb 조회해 선례 추가.

## 배경
- 예선 제안서 제출 완료. 본선 4주 준비 시작.
- Aliphatic_long_chain 완전해결 개선 완료
- 라이브러리 42개 규칙, 선례 11건(5건은 도킹검증 포함)
- 남은 개선 방향: 선례 확장(우선) > 공유결합 도킹(스트레치)

In [2]:
!pip install rdkit -q
!pip install chembl_webresource_client -q
!pip install fuzzywuzzy python-Levenshtein -q
!pip install PyTDC --no-deps -q
!pip install PyYAML tqdm requests -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.4/37.4 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.8/70.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 47.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git clone https://{token}@github.com/Dec32th/laidd-2026.git
%cd /content/laidd-2026

Cloning into 'laidd-2026'...
remote: Enumerating objects: 610, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 610 (delta 32), reused 48 (delta 17), pack-reused 540 (from 1)
Receiving objects: 100% (610/610), 7.14 MiB | 15.66 MiB/s, done.
Resolving deltas: 100% (349/349), done.
/content/laidd-2026


In [4]:
!git config --global user.email "hyekyeong.w@gmail.com"
!git config --global user.name "Dec32th"

In [5]:
import importlib, json
from collections import Counter
from rdkit import Chem
from chembl_webresource_client.new_client import new_client
import requests

import src.tools.replacement_library
import src.tools.molecule_editor
import src.tools.atom_editor
import src.tools.toxicophore_detector
import src.tools.precedent_library

from src.tools.data_prep import load_tox21_clean
from src.tools.toxicophore_detector import detect_toxicophores
from src.tools.replacement_library import get_replacement_candidates
from src.tools.molecule_editor import propose_fix, iterative_fix_loop, clear_failure_memory
from src.tools.precedent_library import PRECEDENT_LIBRARY, get_precedents

data = load_tox21_clean(random_state=7)
molecule = new_client.molecule

base_url = "https://www.guidetopharmacology.org/services"
def search_ligand(name):
    resp = requests.get(f"{base_url}/ligands", params={"name": name})
    return resp.json()
def get_ligand_interactions(ligand_id):
    resp = requests.get(f"{base_url}/ligands/{ligand_id}/interactions")
    return resp.json()

n_rules = len(get_replacement_candidates.__globals__['REPLACEMENT_LIBRARY'])
print(f"라이브러리 규칙 수: {n_rules}, 선례 수: {len(PRECEDENT_LIBRARY)}")

[14:40:58] WARNING: not removing hydrogen atom without neighbors
[14:40:58] Explicit valence for atom # 8 Al, 6, is greater than permitted
[14:40:59] Explicit valence for atom # 3 Al, 6, is greater than permitted
[14:40:59] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:40:59] Explicit valence for atom # 4 Al, 6, is greater than permitted
[14:41:00] Explicit valence for atom # 9 Al, 6, is greater than permitted
[14:41:00] Explicit valence for atom # 5 Al, 6, is greater than permitted
[14:41:00] Explicit valence for atom # 16 Al, 6, is greater than permitted
[14:41:00] Explicit valence for atom # 20 Al, 6, is greater than permitted


전체: 7831개, 파싱 성공: 7823개, 파싱 실패(제외): 8개


[14:41:01] WARNING: not removing hydrogen atom without neighbors


라이브러리 규칙 수: 42, 선례 수: 11


In [6]:
covered_rule_counter = Counter()
for s in data['smiles_valid']:
    p = detect_toxicophores(s)
    for prob in p:
        if get_replacement_candidates(prob['rule_name']) is not None:
            covered_rule_counter[prob['rule_name']] += 1

rules_with_precedent = set(p['rule'] for p in PRECEDENT_LIBRARY)

print("커버된 규칙 중 선례가 없는 것 (빈도순):")
priority_rules = []
for rule, count in covered_rule_counter.most_common(20):
    has_precedent = rule in rules_with_precedent
    marker = "선례있음" if has_precedent else "선례없음 <- 우선순위"
    print(f"  {rule}: {count}건 [{marker}]")
    if not has_precedent:
        priority_rules.append(rule)

print(f"\n우선순위 목록: {priority_rules[:10]}")

커버된 규칙 중 선례가 없는 것 (빈도순):
  Aliphatic_long_chain: 170건 [선례없음 <- 우선순위]
  isolated_alkene: 65건 [선례없음 <- 우선순위]
  alkyl_halide: 58건 [선례있음]
  nitro_group: 53건 [선례없음 <- 우선순위]
  aniline: 43건 [선례없음 <- 우선순위]
  Michael_acceptor_1: 43건 [선례있음]
  Sulfonic_acid_2: 37건 [선례없음 <- 우선순위]
  phosphor: 33건 [선례없음 <- 우선순위]
  quaternary_nitrogen_1: 28건 [선례없음 <- 우선순위]
  aldehyde: 23건 [선례없음 <- 우선순위]
  quaternary_nitrogen_2: 23건 [선례없음 <- 우선순위]
  imine_1_general: 18건 [선례없음 <- 우선순위]
  triple_bond: 17건 [선례없음 <- 우선순위]
  het-C-het_not_in_ring: 15건 [선례없음 <- 우선순위]
  Thiocarbonyl_group: 13건 [선례있음]
  azo_A(324): 13건 [선례있음]
  catechol: 12건 [선례있음]
  beta-keto/anhydride: 11건 [선례있음]
  Three-membered_heterocycle: 10건 [선례없음 <- 우선순위]
  phenol_ester: 9건 [선례없음 <- 우선순위]

우선순위 목록: ['Aliphatic_long_chain', 'isolated_alkene', 'nitro_group', 'aniline', 'Sulfonic_acid_2', 'phosphor', 'quaternary_nitrogen_1', 'aldehyde', 'quaternary_nitrogen_2', 'imine_1_general']


In [7]:
from chembl_webresource_client.new_client import new_client
substructure = new_client.substructure
molecule = new_client.molecule

# Aliphatic_long_chain 규칙을 실제로 고친 형태(사슬 중간에 에테르 산소)가
# 승인 약물 구조에 실제 존재하는지 서브구조 검색으로 확인
query_smiles = "CCCOCCCOCCC"  # 대표 모티프: 프로필렌 단위 사이에 에테르 산소

hits = substructure.filter(smiles=query_smiles)
chembl_ids = [h['molecule_chembl_id'] for h in hits[:300]]
print(f"서브구조 매치 화합물 수: {len(chembl_ids)}")

approved = []
for cid in chembl_ids:
    res = list(molecule.filter(molecule_chembl_id=cid, max_phase=4))
    approved.extend(res)

print(f"\n그중 승인(max_phase=4) 약물: {len(approved)}건")
for a in approved[:30]:
    print(a['molecule_chembl_id'], '|', a.get('pref_name'), '| max_phase=', a.get('max_phase'),
          '| withdrawn=', a.get('withdrawn_flag'), '| smiles=', a.get('molecule_structures', {}).get('canonical_smiles') if a.get('molecule_structures') else None)

서브구조 매치 화합물 수: 300

그중 승인(max_phase=4) 약물: 3건
CHEMBL267345 | AMPHOTERICIN B | max_phase= 4.0 | withdrawn= False | smiles= C[C@@H]1[C@H](O)[C@@H](C)/C=C/C=C/C=C/C=C/C=C/C=C/C=C/[C@H](O[C@@H]2O[C@H](C)[C@@H](O)[C@H](N)[C@@H]2O)C[C@@H]2O[C@](O)(C[C@@H](O)C[C@@H](O)[C@H](O)CC[C@@H](O)C[C@@H](O)CC(=O)O[C@H]1C)C[C@H](O)[C@H]2C(=O)O
CHEMBL529 | AZITHROMYCIN | max_phase= 4.0 | withdrawn= False | smiles= CC[C@H]1OC(=O)[C@H](C)[C@@H](O[C@H]2C[C@@](C)(OC)[C@@H](O)[C@H](C)O2)[C@H](C)[C@@H](O[C@@H]2O[C@H](C)C[C@H](N(C)C)[C@H]2O)[C@](C)(O)C[C@@H](C)CN(C)[C@H](C)[C@@H](O)[C@]1(C)O
CHEMBL532 | ERYTHROMYCIN | max_phase= 4.0 | withdrawn= False | smiles= CC[C@H]1OC(=O)[C@H](C)[C@@H](O[C@H]2C[C@@](C)(OC)[C@@H](O)[C@H](C)O2)[C@H](C)[C@@H](O[C@@H]2O[C@H](C)C[C@H](N(C)C)[C@H]2O)[C@](C)(O)C[C@@H](C)C(=O)[C@H](C)[C@@H](O)[C@]1(C)O


In [8]:
queries = ['CCCOCCCOCCC', 'CCCCOCCCC', 'CCOCCOCCOCC', 'CCCCCOCCCCC']

seen = set()
genuine_hits = []

for q in queries:
    query_mol = Chem.MolFromSmiles(q)
    hits = new_client.substructure.filter(smiles=q)
    chembl_ids = [h['molecule_chembl_id'] for h in hits[:150]]
    for cid in chembl_ids:
        if cid in seen:
            continue
        seen.add(cid)
        res = list(molecule.filter(molecule_chembl_id=cid, max_phase=4))
        if not res:
            continue
        rec = res[0]
        smi = rec.get('molecule_structures', {}).get('canonical_smiles') if rec.get('molecule_structures') else None
        if not smi:
            continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        matches = mol.GetSubstructMatches(query_mol)
        for m in matches:
            if all(not mol.GetAtomWithIdx(i).IsInRing() for i in m):
                genuine_hits.append((rec['molecule_chembl_id'], rec.get('pref_name'), rec.get('withdrawn_flag'), q, smi))
                break

print(f"비고리(열린 사슬) 매치인 승인 약물: {len(genuine_hits)}건")
for h in genuine_hits:
    print(h)

비고리(열린 사슬) 매치인 승인 약물: 0건


In [9]:
from collections import Counter

print(f"총 선례 수: {len(PRECEDENT_LIBRARY)}")

types = Counter(p['type'] for p in PRECEDENT_LIBRARY)
print("\n--- type별 분포 ---")
for t, c in types.most_common():
    print(f"{t}: {c}")

print("\n--- 전체 항목 ---")
for p in PRECEDENT_LIBRARY:
    print(p)

총 선례 수: 11

--- type별 분포 ---
도킹검증_결과: 3
위험=메커니즘_참고: 2
긍정_승인약물쌍: 1
정량_활성데이터: 1
부정_참고사례_검증필요: 1
긍정_통계검증결과: 1
위험=메커니즘_참고_검증완료: 1
도킹검증_방법론한계: 1

--- 전체 항목 ---
{'rule': 'Thiocarbonyl_group', 'type': '긍정_승인약물쌍', 'description': '티오펜탈(C=S)/펜토바비탈(C=O), 티아밀랄(C=S)/세코바비탈(C=O) - 동일 사이드체인, C=S->C=O만 다른 실제 승인 마취제 쌍. baseline 모델 기준 옥소형이 티오형보다 Tox21 평균 예측값 낮음(-0.008~-0.009).'}
{'rule': 'catechol', 'type': '정량_활성데이터', 'description': '도파민이 D1(Ki 4.3-5.6nM)/D2(Ki 4.7-7.2nM)/D3(Ki 6.4-7.3nM) 수용체에 단자릿수 nM 강력 결합 - 카테콜 골격이 활성에 필수적임을 정량적으로 뒷받침.'}
{'rule': 'hydroxamic_acid', 'type': '부정_참고사례_검증필요', 'description': '하이드록삼산 골격(보리노스타트 등 HDAC 억제제)은 아연 킬레이션이 약효 핵심이므로, 이 계열에 대한 무분별한 치환은 약효 상실 위험. (문헌 재확인 필요)'}
{'rule': 'beta-keto/anhydride', 'type': '긍정_통계검증결과', 'description': 'MMPDB 공식 통계 도구로 재검증한 결과, Tox21 규모(1173개)에서 무수물 관련 매칭쌍은 표본 부족(count=1)으로 통계적 유의성 확보 불가 - 데이터형 접근보다 문헌형 근거가 더 신뢰할 만함을 시사.'}
{'rule': 'Michael_acceptor_1', 'type': '위험=메커니즘_참고', 'description': '에타크린산(이뇨제, FDA 승인)은 시스테인 잔기와의 공유결합 자체가 작용 메커니즘인 공유결합 

In [10]:
for name in ['Macrogol', 'Polyethylene glycol', 'PEG 3350']:
    res = search_ligand(name)
    print(name, '->', res)

res = list(molecule.filter(pref_name__icontains='MACROGOL'))
for r in res[:10]:
    print(r['molecule_chembl_id'], r.get('pref_name'), 'max_phase=', r.get('max_phase'), 'withdrawn=', r.get('withdrawn_flag'))
    print(' smiles:', r.get('molecule_structures', {}).get('canonical_smiles') if r.get('molecule_structures') else None)

Macrogol -> {'error': 'No ligands found'}
Polyethylene glycol -> {'error': 'No ligands found'}
PEG 3350 -> {'error': 'No ligands found'}
CHEMBL2109013 CETOMACROGOL 1000 max_phase= 2.0 withdrawn= False
 smiles: None


In [11]:
for name in ['Polidocanol', 'Poloxamer 188', 'Poloxamer 407', 'Nonoxynol']:
    res = search_ligand(name)
    print(name, '->', res)

for term in ['POLIDOCANOL', 'POLOXAMER']:
    res = list(molecule.filter(pref_name__icontains=term))
    for r in res[:10]:
        smi = r.get('molecule_structures', {}).get('canonical_smiles') if r.get('molecule_structures') else None
        print(r['molecule_chembl_id'], r.get('pref_name'), 'max_phase=', r.get('max_phase'), 'withdrawn=', r.get('withdrawn_flag'), 'smiles=', smi)

Polidocanol -> {'error': 'No ligands found'}
Poloxamer 188 -> {'error': 'No ligands found'}
Poloxamer 407 -> {'error': 'No ligands found'}
Nonoxynol -> {'error': 'No ligands found'}
CHEMBL6067961 POLIDOCANOL max_phase= 4.0 withdrawn= False smiles= None
CHEMBL2108134 POLOXAMER max_phase= 3.0 withdrawn= False smiles= None
CHEMBL3137355 VEPOLOXAMER max_phase= 3.0 withdrawn= False smiles= None


In [12]:
rec = molecule.get('CHEMBL6067961')
print(rec.get('pref_name'), rec.get('max_phase'), rec.get('withdrawn_flag'))
print(rec.get('molecule_structures'))

# GtoPdb 쪽도 폴리도카놀 확인
print("\nGtoPdb:")
res = search_ligand('polidocanol')
print(res)

POLIDOCANOL 4.0 False
None

GtoPdb:
{'error': 'No ligands found'}


In [13]:
# 1) 정확한 이름/동의어로 재검색
res = list(molecule.filter(molecule_synonyms__molecule_synonym__iexact='POLIDOCANOL'))
for r in res:
    print(r['molecule_chembl_id'], r.get('pref_name'), r.get('max_phase'))
    print(' structures:', r.get('molecule_structures'))

# 2) InChI/InChIKey라도 있는지 확인
rec = molecule.get('CHEMBL6067961')
print('\n전체 필드 확인:')
for k, v in rec.items():
    print(k, ':', v)

CHEMBL6067961 POLIDOCANOL 4.0
 structures: None

전체 필드 확인:
atc_classifications : ['C05BB02']
availability_type : 1
biotherapeutic : None
chemical_probe : 0
chirality : 1
cross_references : [{'xref_id': 'polidocanol', 'xref_name': 'polidocanol', 'xref_src': 'DailyMed'}]
dosed_ingredient : True
first_approval : 2010
first_in_class : 0
helm_notation : None
inorganic_flag : 0
max_phase : 4.0
molecule_chembl_id : CHEMBL6067961
molecule_hierarchy : {'active_chembl_id': 'CHEMBL6067961', 'molecule_chembl_id': 'CHEMBL6067961', 'parent_chembl_id': 'CHEMBL6067961'}
molecule_properties : None
molecule_structures : None
molecule_synonyms : [{'molecule_synonym': 'Aethoxysklerol', 'syn_type': 'TRADE_NAME', 'synonyms': 'AETHOXYSKLEROL'}, {'molecule_synonym': 'Asclera', 'syn_type': 'TRADE_NAME', 'synonyms': 'ASCLERA'}, {'molecule_synonym': 'Laureth 9', 'syn_type': 'BNF', 'synonyms': 'LAURETH 9'}, {'molecule_synonym': 'Laureth 9', 'syn_type': 'OTHER', 'synonyms': 'LAURETH 9'}, {'molecule_synonym': 'Laur

In [14]:
!cat src/tools/precedent_library.py

"""선례 라이브러리 — 승인/철수 약물, 정량 활성 데이터, 도킹 검증 결과를
판단 에이전트 프롬프트에 실시간 주입하기 위한 구조화된 근거 저장소.
모든 항목은 이 세션에서 ChEMBL/GtoPdb API 조회 또는 실제 도킹 실행으로
직접 확인한 것만 포함한다(추정/일반 지식은 배제).
"""

PRECEDENT_LIBRARY = [
    {"rule": "Thiocarbonyl_group", "type": "긍정_승인약물쌍",
     "description": "티오펜탈(C=S)/펜토바비탈(C=O), 티아밀랄(C=S)/세코바비탈(C=O) - "
                     "동일 사이드체인, C=S->C=O만 다른 실제 승인 마취제 쌍. "
                     "baseline 모델 기준 옥소형이 티오형보다 Tox21 평균 예측값 낮음(-0.008~-0.009)."},
    {"rule": "catechol", "type": "정량_활성데이터",
     "description": "도파민이 D1(Ki 4.3-5.6nM)/D2(Ki 4.7-7.2nM)/D3(Ki 6.4-7.3nM) 수용체에 "
                     "단자릿수 nM 강력 결합 - 카테콜 골격이 활성에 필수적임을 정량적으로 뒷받침."},
    {"rule": "hydroxamic_acid", "type": "부정_참고사례_검증필요",
     "description": "하이드록삼산 골격(보리노스타트 등 HDAC 억제제)은 아연 킬레이션이 "
                     "약효 핵심이므로, 이 계열에 대한 무분별한 치환은 약효 상실 위험. "
                     "(문헌 재확인 필요)"},
    {"rule": "beta-keto/anhydride", "type": "긍정_통계검증결과",
     "description": "MMPDB 공식 통계 도구로 재검증한 결과, Tox21 규모(1173개)에서 "
     

In [16]:
%%writefile src/tools/precedent_library.py


"""선례 라이브러리 — 승인/철수 약물, 정량 활성 데이터, 도킹 검증 결과를
판단 에이전트 프롬프트에 실시간 주입하기 위한 구조화된 근거 저장소.
모든 항목은 이 세션에서 ChEMBL/GtoPdb API 조회 또는 실제 도킹 실행으로
직접 확인한 것만 포함한다(추정/일반 지식은 배제).
"""

PRECEDENT_LIBRARY = [
    {"rule": "Thiocarbonyl_group", "type": "긍정_승인약물쌍",
     "description": "티오펜탈(C=S)/펜토바비탈(C=O), 티아밀랄(C=S)/세코바비탈(C=O) - "
                     "동일 사이드체인, C=S->C=O만 다른 실제 승인 마취제 쌍. "
                     "baseline 모델 기준 옥소형이 티오형보다 Tox21 평균 예측값 낮음(-0.008~-0.009)."},
    {"rule": "catechol", "type": "정량_활성데이터",
     "description": "도파민이 D1(Ki 4.3-5.6nM)/D2(Ki 4.7-7.2nM)/D3(Ki 6.4-7.3nM) 수용체에 "
                     "단자릿수 nM 강력 결합 - 카테콜 골격이 활성에 필수적임을 정량적으로 뒷받침."},
    {"rule": "hydroxamic_acid", "type": "부정_참고사례_검증필요",
     "description": "하이드록삼산 골격(보리노스타트 등 HDAC 억제제)은 아연 킬레이션이 "
                     "약효 핵심이므로, 이 계열에 대한 무분별한 치환은 약효 상실 위험. "
                     "(문헌 재확인 필요)"},
    {"rule": "beta-keto/anhydride", "type": "긍정_통계검증결과",
     "description": "MMPDB 공식 통계 도구로 재검증한 결과, Tox21 규모(1173개)에서 "
                     "무수물 관련 매칭쌍은 표본 부족(count=1)으로 통계적 유의성 확보 불가 - "
                     "데이터형 접근보다 문헌형 근거가 더 신뢰할 만함을 시사."},
    {"rule": "Michael_acceptor_1", "type": "위험=메커니즘_참고",
     "description": "에타크린산(이뇨제, FDA 승인)은 시스테인 잔기와의 공유결합 자체가 "
                     "작용 메커니즘인 공유결합 억제제 - Michael acceptor 경고가 항상 "
                     "제거 대상은 아님을 보여주는 실제 승인약물 사례."},
    {"rule": "alkyl_halide", "type": "위험=메커니즘_참고",
     "description": "메클로르에타민, 사이클로포스파미드 등 알킬화 항암제는 DNA 알킬화 "
                     "반응성 자체가 세포독성 치료 메커니즘 - 이 계열에는 할로겐 제거가 "
                     "부적절함을 보여주는 실제 승인약물 사례."},
    {"rule": "azo_A(324)", "type": "위험=메커니즘_참고_검증완료",
     "description": "설파살라진(SMILES 내 /N=N/ 아조 결합 확인, ChEMBL max_phase=4.0, "
                     "GtoPdb FDA 승인 1950년/WHO 필수의약품)은 아조 결합이 장내 "
                     "세균에 의해 환원되어 활성 대사물(5-ASA)을 방출하는 프로드러그 - "
                     "실제 조회로 검증됨."},
    {"rule": "catechol", "type": "도킹검증_결과",
     "description": "COMT(PDB 1VID) 도킹 검증: 도파민(-5.72 kcal/mol)→메톡시도파민"
                     "(-5.41 kcal/mol), 변화폭 +0.31 kcal/mol로 약화 방향이나 이는 "
                     "1 kcal/mol 미만의 작은 차이로 도킹 자체의 오차범위 내일 수 있어 "
                     "단정적 근거로 삼기엔 약함. 에피네프린은 반대로 미세 강화"
                     "(-6.21→-6.32, -0.10) - 두 경우 모두 변화폭이 작아, 도킹 수치보다는 "
                     "카테콜의 수용체 결합 필수성(정성적 근거)이 더 강한 판단 기준."},
    {"rule": "Michael_acceptor_1", "type": "도킹검증_방법론한계",
     "description": "EGFR(PDB 6JX4) 도킹 검증: 오시메르티닙(-7.13)→C=C환원버전(-7.08), "
                     "거의 무변화(+0.05). 표준(비공유) 도킹이 오시메르티닙의 실제 "
                     "공유결합(Cys797) 메커니즘을 포착하지 못하는 방법론적 한계 확인 - "
                     "공유결합 억제제 계열은 일반 도킹 스코어만으로 활성 손실을 판단하지 "
                     "말 것(도킹 무변화가 곧 활성 유지를 뜻하지 않음)."},
    {"rule": "hydroquinone", "type": "도킹검증_결과",
     "description": "NQO1 도킹 검증: 퀴논(-3.29)→하이드로퀴논(-4.08), 결합 강화(-0.79, "
                     "1 kcal/mol에 근접하는 뚜렷한 변화). 메틸퀴논(-3.80)→환원버전"
                     "(-4.29)도 강화(-0.49), 2건 모두 일관되게 강화 방향. NQO1이 실제로 "
                     "퀴논을 하이드로퀴논으로 환원하는 효소이므로, 이 치환 방향은 해독 "
                     "반응경로와 자연스럽게 정렬되며 실측 결합력도 개선됨 - 활성 손실 "
                     "우려가 낮은 것으로 확인됨."},
    {"rule": "quinone_A(370)", "type": "도킹검증_결과",
     "description": "NQO1 도킹 검증: 퀴논(-3.29)→하이드로퀴논(-4.08), 결합 강화(-0.79). "
                     "실제 표적 효소와의 결합력이 오히려 개선되는 것으로 실측 확인됨 "
                     "(hydroquinone 규칙과 동일 표적 데이터 공유)."},
    {"rule": "Aliphatic_long_chain", "type": "긍정_승인약물_확인(구조는_동의어로_대체확인)",
     "description": "POLIDOCANOL(라우릴알코올+에틸렌옥사이드 평균 9개 반복부가체)은 ChEMBL 조회로 "
                     "승인 확인됨(max_phase=4.0, first_approval=2010, ATC C05BB02, "
                     "dosed_ingredient=True, withdrawn=False, 상품명 Asclera/Aethoxysklerol). "
                     "ChEMBL에 단일 SMILES는 없으나(polymer_flag=1, structure_type=NONE) "
                     "이는 다분산 고분자라 원천적으로 단일 구조가 없기 때문이며, 공식 동의어"
                     "(USP: Polyoxyl 9 lauryl ether, JAN: Lauromacrogol 400)가 "
                     "\"장쇄 알킬+반복 에테르\" 구조를 명확히 정의함 - 실제 승인약물에서 "
                     "이 전략이 쓰이고 있음을 뒷받침."},
    {"rule": "Aliphatic_long_chain", "type": "부정_참고사례_검증필요",
     "description": "ChEMBL 서브구조 검색(에테르 삽입 사슬 모티프)으로 매치된 승인약물은 "
                     "에리스로마이신/아지스로마이신/암포테리신B였으나, 매치 위치를 IsInRing으로 "
                     "확인한 결과 전부 매크로락톤/당 고리 내부의 고리형 에테르로, 우리 규칙이 "
                     "다루는 \"고리 밖 열린 사슬\" 상황과는 구조적으로 다름 - 이 계열은 직접적 "
                     "근거로 부적합함이 확인됨."},
]


def get_precedents(rule_name: str) -> str | None:
    """규칙 이름으로 관련 선례를 찾아 프롬프트에 넣을 텍스트로 반환."""
    matches = [p for p in PRECEDENT_LIBRARY if p['rule'] == rule_name]
    if not matches:
        return None
    return "\n".join([f"- [{m['type']}] {m['description']}" for m in matches])


Overwriting src/tools/precedent_library.py


In [17]:
importlib.reload(src.tools.precedent_library)
from src.tools.precedent_library import PRECEDENT_LIBRARY, get_precedents
clear_failure_memory()

print(f"총 선례 수: {len(PRECEDENT_LIBRARY)} (13이어야 정상)")
print(get_precedents('Aliphatic_long_chain'))

총 선례 수: 13 (13이어야 정상)
- [긍정_승인약물_확인(구조는_동의어로_대체확인)] POLIDOCANOL(라우릴알코올+에틸렌옥사이드 평균 9개 반복부가체)은 ChEMBL 조회로 승인 확인됨(max_phase=4.0, first_approval=2010, ATC C05BB02, dosed_ingredient=True, withdrawn=False, 상품명 Asclera/Aethoxysklerol). ChEMBL에 단일 SMILES는 없으나(polymer_flag=1, structure_type=NONE) 이는 다분산 고분자라 원천적으로 단일 구조가 없기 때문이며, 공식 동의어(USP: Polyoxyl 9 lauryl ether, JAN: Lauromacrogol 400)가 "장쇄 알킬+반복 에테르" 구조를 명확히 정의함 - 실제 승인약물에서 이 전략이 쓰이고 있음을 뒷받침.
- [부정_참고사례_검증필요] ChEMBL 서브구조 검색(에테르 삽입 사슬 모티프)으로 매치된 승인약물은 에리스로마이신/아지스로마이신/암포테리신B였으나, 매치 위치를 IsInRing으로 확인한 결과 전부 매크로락톤/당 고리 내부의 고리형 에테르로, 우리 규칙이 다루는 "고리 밖 열린 사슬" 상황과는 구조적으로 다름 - 이 계열은 직접적 근거로 부적합함이 확인됨.


In [18]:
!cd /content/laidd-2026 && git add . && git commit -m "Add Aliphatic_long_chain precedents to PRECEDENT_LIBRARY (POLIDOCANOL approved-drug evidence + macrolide non-fit note)" && git push

[main 875e3c4] Add Aliphatic_long_chain precedents to PRECEDENT_LIBRARY (POLIDOCANOL approved-drug evidence + macrolide non-fit note)
 1 file changed, 17 insertions(+)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.36 KiB | 1.36 MiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   c458cb2..875e3c4  main -> main


In [19]:
info = get_replacement_candidates('isolated_alkene')
print("problem_smarts:", info['problem_smarts'])
for i, c in enumerate(info['candidates']):
    print(f"\ncandidate idx={i}")
    for k, v in c.items():
        print(f"  {k}: {v}")

problem_smarts: [CX3H1,CX3H0;!$([CX3]=[CX3]c)]=[CX3;!$([CX3]=[CX3]c)]

candidate idx=0
  edit_type: reduce_bond
  name: saturated (C-C single bond)
  rationale: 고립된 지방족 알켄(방향족·카르보닐과 공액되지 않은 단순 C=C)은 산화적 대사(에폭시드 형성 등)를 거쳐 반응성 중간체를 생성할 가능성이 있는 구조 경고임. 이중결합을 단일결합으로 환원해 이 산화 경로를 차단함 (검증 필요)


In [20]:
from rdkit import Chem

pattern = Chem.MolFromSmarts(info['problem_smarts'])

query_candidates = ['CCC=CCC', 'CCCC=CCCC', 'CC=CCC']

seen = set()
hits_found = []

for q in query_candidates:
    hits = new_client.substructure.filter(smiles=q)
    chembl_ids = [h['molecule_chembl_id'] for h in hits[:150]]
    for cid in chembl_ids:
        if cid in seen:
            continue
        seen.add(cid)
        res = list(molecule.filter(molecule_chembl_id=cid, max_phase=4))
        if not res:
            continue
        rec = res[0]
        smi = rec.get('molecule_structures', {}).get('canonical_smiles') if rec.get('molecule_structures') else None
        if not smi:
            continue
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            continue
        if mol.HasSubstructMatch(pattern):
            hits_found.append((rec['molecule_chembl_id'], rec.get('pref_name'), rec.get('withdrawn_flag'), smi))

print(f"실제 problem_smarts 매치인 승인약물: {len(hits_found)}건")
for h in hits_found[:30]:
    print(h)

실제 problem_smarts 매치인 승인약물: 5건
('CHEMBL413', 'SIROLIMUS', False, 'CO[C@H]1C[C@@H]2CC[C@@H](C)[C@@](O)(O2)C(=O)C(=O)N2CCCC[C@H]2C(=O)O[C@H]([C@H](C)C[C@@H]2CC[C@@H](O)[C@H](OC)C2)CC(=O)[C@H](C)/C=C(\\C)[C@@H](O)[C@@H](OC)C(=O)[C@H](C)C[C@H](C)/C=C/C=C/C=C/1C')
('CHEMBL269732', 'TACROLIMUS ANHYDROUS', False, 'C=CC[C@@H]1/C=C(\\C)C[C@H](C)C[C@H](OC)[C@H]2O[C@@](O)(C(=O)C(=O)N3CCCC[C@H]3C(=O)O[C@H](/C(C)=C/[C@@H]3CC[C@@H](O)[C@H](OC)C3)[C@H](C)[C@@H](O)CC1=O)[C@H](C)C[C@@H]2OC')
('CHEMBL268164', 'CYCLOBARBITAL', True, 'CCC1(C2=CCCCC2)C(=O)NC(=O)NC1=O')
('CHEMBL38', 'TRETINOIN', False, 'CC1=C(/C=C/C(C)=C/C=C/C(C)=C/C(=O)O)C(C)(C)CCC1')
('CHEMBL7728', 'HEXOBARBITAL', True, 'CN1C(=O)NC(=O)C(C)(C2=CCCCC2)C1=O')


In [21]:
%%writefile src/tools/precedent_library.py


"""선례 라이브러리 — 승인/철수 약물, 정량 활성 데이터, 도킹 검증 결과를
판단 에이전트 프롬프트에 실시간 주입하기 위한 구조화된 근거 저장소.
모든 항목은 이 세션에서 ChEMBL/GtoPdb API 조회 또는 실제 도킹 실행으로
직접 확인한 것만 포함한다(추정/일반 지식은 배제).
"""

PRECEDENT_LIBRARY = [
    {"rule": "Thiocarbonyl_group", "type": "긍정_승인약물쌍",
     "description": "티오펜탈(C=S)/펜토바비탈(C=O), 티아밀랄(C=S)/세코바비탈(C=O) - "
                     "동일 사이드체인, C=S->C=O만 다른 실제 승인 마취제 쌍. "
                     "baseline 모델 기준 옥소형이 티오형보다 Tox21 평균 예측값 낮음(-0.008~-0.009)."},
    {"rule": "catechol", "type": "정량_활성데이터",
     "description": "도파민이 D1(Ki 4.3-5.6nM)/D2(Ki 4.7-7.2nM)/D3(Ki 6.4-7.3nM) 수용체에 "
                     "단자릿수 nM 강력 결합 - 카테콜 골격이 활성에 필수적임을 정량적으로 뒷받침."},
    {"rule": "hydroxamic_acid", "type": "부정_참고사례_검증필요",
     "description": "하이드록삼산 골격(보리노스타트 등 HDAC 억제제)은 아연 킬레이션이 "
                     "약효 핵심이므로, 이 계열에 대한 무분별한 치환은 약효 상실 위험. "
                     "(문헌 재확인 필요)"},
    {"rule": "beta-keto/anhydride", "type": "긍정_통계검증결과",
     "description": "MMPDB 공식 통계 도구로 재검증한 결과, Tox21 규모(1173개)에서 "
                     "무수물 관련 매칭쌍은 표본 부족(count=1)으로 통계적 유의성 확보 불가 - "
                     "데이터형 접근보다 문헌형 근거가 더 신뢰할 만함을 시사."},
    {"rule": "Michael_acceptor_1", "type": "위험=메커니즘_참고",
     "description": "에타크린산(이뇨제, FDA 승인)은 시스테인 잔기와의 공유결합 자체가 "
                     "작용 메커니즘인 공유결합 억제제 - Michael acceptor 경고가 항상 "
                     "제거 대상은 아님을 보여주는 실제 승인약물 사례."},
    {"rule": "alkyl_halide", "type": "위험=메커니즘_참고",
     "description": "메클로르에타민, 사이클로포스파미드 등 알킬화 항암제는 DNA 알킬화 "
                     "반응성 자체가 세포독성 치료 메커니즘 - 이 계열에는 할로겐 제거가 "
                     "부적절함을 보여주는 실제 승인약물 사례."},
    {"rule": "azo_A(324)", "type": "위험=메커니즘_참고_검증완료",
     "description": "설파살라진(SMILES 내 /N=N/ 아조 결합 확인, ChEMBL max_phase=4.0, "
                     "GtoPdb FDA 승인 1950년/WHO 필수의약품)은 아조 결합이 장내 "
                     "세균에 의해 환원되어 활성 대사물(5-ASA)을 방출하는 프로드러그 - "
                     "실제 조회로 검증됨."},
    {"rule": "catechol", "type": "도킹검증_결과",
     "description": "COMT(PDB 1VID) 도킹 검증: 도파민(-5.72 kcal/mol)→메톡시도파민"
                     "(-5.41 kcal/mol), 변화폭 +0.31 kcal/mol로 약화 방향이나 이는 "
                     "1 kcal/mol 미만의 작은 차이로 도킹 자체의 오차범위 내일 수 있어 "
                     "단정적 근거로 삼기엔 약함. 에피네프린은 반대로 미세 강화"
                     "(-6.21→-6.32, -0.10) - 두 경우 모두 변화폭이 작아, 도킹 수치보다는 "
                     "카테콜의 수용체 결합 필수성(정성적 근거)이 더 강한 판단 기준."},
    {"rule": "Michael_acceptor_1", "type": "도킹검증_방법론한계",
     "description": "EGFR(PDB 6JX4) 도킹 검증: 오시메르티닙(-7.13)→C=C환원버전(-7.08), "
                     "거의 무변화(+0.05). 표준(비공유) 도킹이 오시메르티닙의 실제 "
                     "공유결합(Cys797) 메커니즘을 포착하지 못하는 방법론적 한계 확인 - "
                     "공유결합 억제제 계열은 일반 도킹 스코어만으로 활성 손실을 판단하지 "
                     "말 것(도킹 무변화가 곧 활성 유지를 뜻하지 않음)."},
    {"rule": "hydroquinone", "type": "도킹검증_결과",
     "description": "NQO1 도킹 검증: 퀴논(-3.29)→하이드로퀴논(-4.08), 결합 강화(-0.79, "
                     "1 kcal/mol에 근접하는 뚜렷한 변화). 메틸퀴논(-3.80)→환원버전"
                     "(-4.29)도 강화(-0.49), 2건 모두 일관되게 강화 방향. NQO1이 실제로 "
                     "퀴논을 하이드로퀴논으로 환원하는 효소이므로, 이 치환 방향은 해독 "
                     "반응경로와 자연스럽게 정렬되며 실측 결합력도 개선됨 - 활성 손실 "
                     "우려가 낮은 것으로 확인됨."},
    {"rule": "quinone_A(370)", "type": "도킹검증_결과",
     "description": "NQO1 도킹 검증: 퀴논(-3.29)→하이드로퀴논(-4.08), 결합 강화(-0.79). "
                     "실제 표적 효소와의 결합력이 오히려 개선되는 것으로 실측 확인됨 "
                     "(hydroquinone 규칙과 동일 표적 데이터 공유)."},
    {"rule": "Aliphatic_long_chain", "type": "긍정_승인약물_확인(구조는_동의어로_대체확인)",
     "description": "POLIDOCANOL(라우릴알코올+에틸렌옥사이드 평균 9개 반복부가체)은 ChEMBL 조회로 "
                     "승인 확인됨(max_phase=4.0, first_approval=2010, ATC C05BB02, "
                     "dosed_ingredient=True, withdrawn=False, 상품명 Asclera/Aethoxysklerol). "
                     "ChEMBL에 단일 SMILES는 없으나(polymer_flag=1, structure_type=NONE) "
                     "이는 다분산 고분자라 원천적으로 단일 구조가 없기 때문이며, 공식 동의어"
                     "(USP: Polyoxyl 9 lauryl ether, JAN: Lauromacrogol 400)가 "
                     "\"장쇄 알킬+반복 에테르\" 구조를 명확히 정의함 - 실제 승인약물에서 "
                     "이 전략이 쓰이고 있음을 뒷받침."},
    {"rule": "Aliphatic_long_chain", "type": "부정_참고사례_검증필요",
     "description": "ChEMBL 서브구조 검색(에테르 삽입 사슬 모티프)으로 매치된 승인약물은 "
                     "에리스로마이신/아지스로마이신/암포테리신B였으나, 매치 위치를 IsInRing으로 "
                     "확인한 결과 전부 매크로락톤/당 고리 내부의 고리형 에테르로, 우리 규칙이 "
                     "다루는 \"고리 밖 열린 사슬\" 상황과는 구조적으로 다름 - 이 계열은 직접적 "
                     "근거로 부적합함이 확인됨."},
    {"rule": "isolated_alkene", "type": "긍정_승인약물쌍",
     "description": "SIROLIMUS(시롤리무스), TACROLIMUS ANHYDROUS(타크로리무스) - 둘 다 ChEMBL "
                     "조회로 승인·비철수 확인됨(withdrawn_flag=False), 대형 매크로라이드 면역억제제로 "
                     "현재도 널리 처방됨. problem_smarts로 직접 매치되는 고립 지방족 알켄이 구조 "
                     "안에 실제 존재 - 고립 알켄이 항상 제거 대상은 아님을 보여주는 실제 승인약물 사례."},
    {"rule": "isolated_alkene", "type": "위험=메커니즘_참고_인과불명",
     "description": "CYCLOBARBITAL, HEXOBARBITAL 둘 다 ChEMBL 조회로 withdrawn_flag=True 확인됨, "
                     "둘 다 problem_smarts에 매치되는 사이클로헥세닐 고립 알켄 치환기를 가짐. "
                     "다만 바르비투르산염 계열은 호흡억제·의존성 등 일반적 안전성 문제로 철수된 "
                     "사례가 많아, 이 알켄 구조가 철수의 직접 원인이라는 인과관계는 확인되지 않음 "
                     "(상관관계만 관찰, 문헌 추가 확인 필요)."},
]


def get_precedents(rule_name: str) -> str | None:
    """규칙 이름으로 관련 선례를 찾아 프롬프트에 넣을 텍스트로 반환."""
    matches = [p for p in PRECEDENT_LIBRARY if p['rule'] == rule_name]
    if not matches:
        return None
    return "\n".join([f"- [{m['type']}] {m['description']}" for m in matches])


Overwriting src/tools/precedent_library.py


In [22]:
importlib.reload(src.tools.precedent_library)
from src.tools.precedent_library import PRECEDENT_LIBRARY, get_precedents
clear_failure_memory()

print(f"총 선례 수: {len(PRECEDENT_LIBRARY)} (15여야 정상)")
print(get_precedents('isolated_alkene'))

총 선례 수: 15 (15여야 정상)
- [긍정_승인약물쌍] SIROLIMUS(시롤리무스), TACROLIMUS ANHYDROUS(타크로리무스) - 둘 다 ChEMBL 조회로 승인·비철수 확인됨(withdrawn_flag=False), 대형 매크로라이드 면역억제제로 현재도 널리 처방됨. problem_smarts로 직접 매치되는 고립 지방족 알켄이 구조 안에 실제 존재 - 고립 알켄이 항상 제거 대상은 아님을 보여주는 실제 승인약물 사례.
- [위험=메커니즘_참고_인과불명] CYCLOBARBITAL, HEXOBARBITAL 둘 다 ChEMBL 조회로 withdrawn_flag=True 확인됨, 둘 다 problem_smarts에 매치되는 사이클로헥세닐 고립 알켄 치환기를 가짐. 다만 바르비투르산염 계열은 호흡억제·의존성 등 일반적 안전성 문제로 철수된 사례가 많아, 이 알켄 구조가 철수의 직접 원인이라는 인과관계는 확인되지 않음 (상관관계만 관찰, 문헌 추가 확인 필요).


In [23]:
import ast

# 1) 문법 검증 (파일 자체가 파싱 가능한지)
with open('src/tools/precedent_library.py') as f:
    source = f.read()
try:
    ast.parse(source)
    print("✅ precedent_library.py 문법 정상")
except SyntaxError as e:
    print("❌ 문법 오류:", e)

# 2) 재로드 후 실제 import/실행 확인
importlib.reload(src.tools.precedent_library)
from src.tools.precedent_library import PRECEDENT_LIBRARY, get_precedents

assert len(PRECEDENT_LIBRARY) == 15, f"개수 불일치: {len(PRECEDENT_LIBRARY)}"
print(f"✅ 총 {len(PRECEDENT_LIBRARY)}건")

# 3) 각 항목이 필수 키(rule, type, description)를 다 갖고 있는지
required_keys = {'rule', 'type', 'description'}
for i, p in enumerate(PRECEDENT_LIBRARY):
    missing = required_keys - p.keys()
    assert not missing, f"{i}번째 항목에 키 누락: {missing}"
print("✅ 모든 항목 필수 키 정상")

# 4) 새로 추가한 두 규칙이 제대로 조회되는지
for rule in ['Aliphatic_long_chain', 'isolated_alkene']:
    result = get_precedents(rule)
    assert result is not None, f"{rule} 조회 실패"
    print(f"✅ {rule}: {result.count(chr(10))+1}줄 반환")

# 5) 다른 프로세스(iterative_fix_loop 등)에 영향 없는지 간단 스모크 테스트
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory
clear_failure_memory()
smoke_result = iterative_fix_loop('CCCCCCCCCCCCCCCC', max_iterations=10, candidate_idx=0)
assert smoke_result['status'] == 'success', f"스모크 테스트 실패: {smoke_result['status']}"
print("✅ iterative_fix_loop 스모크 테스트 통과")

print("\n전체 통과 — 커밋해도 안전합니다.")

✅ precedent_library.py 문법 정상
✅ 총 15건
✅ 모든 항목 필수 키 정상
✅ Aliphatic_long_chain: 2줄 반환
✅ isolated_alkene: 2줄 반환
✅ iterative_fix_loop 스모크 테스트 통과

전체 통과 — 커밋해도 안전합니다.


In [24]:
!cd /content/laidd-2026 && git add . && git commit -m "Add isolated_alkene precedents to PRECEDENT_LIBRARY (sirolimus/tacrolimus approved-drug evidence + barbiturate withdrawn note)" && git push

[main 50eeba7] Add isolated_alkene precedents to PRECEDENT_LIBRARY (sirolimus/tacrolimus approved-drug evidence + barbiturate withdrawn note)
 1 file changed, 11 insertions(+)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 1.10 KiB | 1.10 MiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   875e3c4..50eeba7  main -> main


In [25]:
info = get_replacement_candidates('nitro_group')
print("problem_smarts:", info['problem_smarts'])
for i, c in enumerate(info['candidates']):
    print(f"\ncandidate idx={i}")
    for k, v in c.items():
        print(f"  {k}: {v}")

problem_smarts: [N+](=O)[O-]

candidate idx=0
  smiles: N
  name: primary amine
  rationale: [참고] 메트로니다졸, 니트로푸란토인, 벤즈니다졸 등 일부 항균제/항기생충제는 니트로기의 선택적 환원 활성화 자체가 치료 메커니즘이므로, 이런 프로드러그 설계 맥락에서는 본 치환이 적절하지 않을 수 있음. || 극성을 유지하면서 니트로기의 환원성 대사 중간체 생성 경로를 제거함

candidate idx=1
  smiles: S(=O)(=O)N
  name: sulfonamide
  rationale: 약물유사 골격에서 흔히 쓰이는 안정적 대체기로, 수소결합 donor/acceptor 특성을 일부 유지

candidate idx=2
  smiles: C#N
  name: nitrile
  rationale: 대사 안정성이 개선된 사례가 문헌에 다수 보고됨, 다만 극성은 다소 감소


In [26]:
for name in ['METRONIDAZOLE', 'NITROFURANTOIN', 'BENZNIDAZOLE']:
    res = list(molecule.filter(pref_name__iexact=name))
    for r in res[:3]:
        print(r['molecule_chembl_id'], r.get('pref_name'), 'max_phase=', r.get('max_phase'), 'withdrawn=', r.get('withdrawn_flag'))
        smi = r.get('molecule_structures', {}).get('canonical_smiles') if r.get('molecule_structures') else None
        print('  smiles:', smi)

CHEMBL137 METRONIDAZOLE max_phase= 4.0 withdrawn= False
  smiles: Cc1ncc([N+](=O)[O-])n1CCO
CHEMBL572 NITROFURANTOIN max_phase= 4.0 withdrawn= False
  smiles: O=C1CN(/N=C/c2ccc([N+](=O)[O-])o2)C(=O)N1
CHEMBL110 BENZNIDAZOLE max_phase= 4.0 withdrawn= False
  smiles: O=C(Cn1ccnc1[N+](=O)[O-])NCc1ccccc1


In [27]:
%%writefile src/tools/precedent_library.py


"""선례 라이브러리 — 승인/철수 약물, 정량 활성 데이터, 도킹 검증 결과를
판단 에이전트 프롬프트에 실시간 주입하기 위한 구조화된 근거 저장소.
모든 항목은 이 세션에서 ChEMBL/GtoPdb API 조회 또는 실제 도킹 실행으로
직접 확인한 것만 포함한다(추정/일반 지식은 배제).
"""

PRECEDENT_LIBRARY = [
    {"rule": "Thiocarbonyl_group", "type": "긍정_승인약물쌍",
     "description": "티오펜탈(C=S)/펜토바비탈(C=O), 티아밀랄(C=S)/세코바비탈(C=O) - "
                     "동일 사이드체인, C=S->C=O만 다른 실제 승인 마취제 쌍. "
                     "baseline 모델 기준 옥소형이 티오형보다 Tox21 평균 예측값 낮음(-0.008~-0.009)."},
    {"rule": "catechol", "type": "정량_활성데이터",
     "description": "도파민이 D1(Ki 4.3-5.6nM)/D2(Ki 4.7-7.2nM)/D3(Ki 6.4-7.3nM) 수용체에 "
                     "단자릿수 nM 강력 결합 - 카테콜 골격이 활성에 필수적임을 정량적으로 뒷받침."},
    {"rule": "hydroxamic_acid", "type": "부정_참고사례_검증필요",
     "description": "하이드록삼산 골격(보리노스타트 등 HDAC 억제제)은 아연 킬레이션이 "
                     "약효 핵심이므로, 이 계열에 대한 무분별한 치환은 약효 상실 위험. "
                     "(문헌 재확인 필요)"},
    {"rule": "beta-keto/anhydride", "type": "긍정_통계검증결과",
     "description": "MMPDB 공식 통계 도구로 재검증한 결과, Tox21 규모(1173개)에서 "
                     "무수물 관련 매칭쌍은 표본 부족(count=1)으로 통계적 유의성 확보 불가 - "
                     "데이터형 접근보다 문헌형 근거가 더 신뢰할 만함을 시사."},
    {"rule": "Michael_acceptor_1", "type": "위험=메커니즘_참고",
     "description": "에타크린산(이뇨제, FDA 승인)은 시스테인 잔기와의 공유결합 자체가 "
                     "작용 메커니즘인 공유결합 억제제 - Michael acceptor 경고가 항상 "
                     "제거 대상은 아님을 보여주는 실제 승인약물 사례."},
    {"rule": "alkyl_halide", "type": "위험=메커니즘_참고",
     "description": "메클로르에타민, 사이클로포스파미드 등 알킬화 항암제는 DNA 알킬화 "
                     "반응성 자체가 세포독성 치료 메커니즘 - 이 계열에는 할로겐 제거가 "
                     "부적절함을 보여주는 실제 승인약물 사례."},
    {"rule": "azo_A(324)", "type": "위험=메커니즘_참고_검증완료",
     "description": "설파살라진(SMILES 내 /N=N/ 아조 결합 확인, ChEMBL max_phase=4.0, "
                     "GtoPdb FDA 승인 1950년/WHO 필수의약품)은 아조 결합이 장내 "
                     "세균에 의해 환원되어 활성 대사물(5-ASA)을 방출하는 프로드러그 - "
                     "실제 조회로 검증됨."},
    {"rule": "catechol", "type": "도킹검증_결과",
     "description": "COMT(PDB 1VID) 도킹 검증: 도파민(-5.72 kcal/mol)→메톡시도파민"
                     "(-5.41 kcal/mol), 변화폭 +0.31 kcal/mol로 약화 방향이나 이는 "
                     "1 kcal/mol 미만의 작은 차이로 도킹 자체의 오차범위 내일 수 있어 "
                     "단정적 근거로 삼기엔 약함. 에피네프린은 반대로 미세 강화"
                     "(-6.21→-6.32, -0.10) - 두 경우 모두 변화폭이 작아, 도킹 수치보다는 "
                     "카테콜의 수용체 결합 필수성(정성적 근거)이 더 강한 판단 기준."},
    {"rule": "Michael_acceptor_1", "type": "도킹검증_방법론한계",
     "description": "EGFR(PDB 6JX4) 도킹 검증: 오시메르티닙(-7.13)→C=C환원버전(-7.08), "
                     "거의 무변화(+0.05). 표준(비공유) 도킹이 오시메르티닙의 실제 "
                     "공유결합(Cys797) 메커니즘을 포착하지 못하는 방법론적 한계 확인 - "
                     "공유결합 억제제 계열은 일반 도킹 스코어만으로 활성 손실을 판단하지 "
                     "말 것(도킹 무변화가 곧 활성 유지를 뜻하지 않음)."},
    {"rule": "hydroquinone", "type": "도킹검증_결과",
     "description": "NQO1 도킹 검증: 퀴논(-3.29)→하이드로퀴논(-4.08), 결합 강화(-0.79, "
                     "1 kcal/mol에 근접하는 뚜렷한 변화). 메틸퀴논(-3.80)→환원버전"
                     "(-4.29)도 강화(-0.49), 2건 모두 일관되게 강화 방향. NQO1이 실제로 "
                     "퀴논을 하이드로퀴논으로 환원하는 효소이므로, 이 치환 방향은 해독 "
                     "반응경로와 자연스럽게 정렬되며 실측 결합력도 개선됨 - 활성 손실 "
                     "우려가 낮은 것으로 확인됨."},
    {"rule": "quinone_A(370)", "type": "도킹검증_결과",
     "description": "NQO1 도킹 검증: 퀴논(-3.29)→하이드로퀴논(-4.08), 결합 강화(-0.79). "
                     "실제 표적 효소와의 결합력이 오히려 개선되는 것으로 실측 확인됨 "
                     "(hydroquinone 규칙과 동일 표적 데이터 공유)."},
    {"rule": "Aliphatic_long_chain", "type": "긍정_승인약물_확인(구조는_동의어로_대체확인)",
     "description": "POLIDOCANOL(라우릴알코올+에틸렌옥사이드 평균 9개 반복부가체)은 ChEMBL 조회로 "
                     "승인 확인됨(max_phase=4.0, first_approval=2010, ATC C05BB02, "
                     "dosed_ingredient=True, withdrawn=False, 상품명 Asclera/Aethoxysklerol). "
                     "ChEMBL에 단일 SMILES는 없으나(polymer_flag=1, structure_type=NONE) "
                     "이는 다분산 고분자라 원천적으로 단일 구조가 없기 때문이며, 공식 동의어"
                     "(USP: Polyoxyl 9 lauryl ether, JAN: Lauromacrogol 400)가 "
                     "\"장쇄 알킬+반복 에테르\" 구조를 명확히 정의함 - 실제 승인약물에서 "
                     "이 전략이 쓰이고 있음을 뒷받침."},
    {"rule": "Aliphatic_long_chain", "type": "부정_참고사례_검증필요",
     "description": "ChEMBL 서브구조 검색(에테르 삽입 사슬 모티프)으로 매치된 승인약물은 "
                     "에리스로마이신/아지스로마이신/암포테리신B였으나, 매치 위치를 IsInRing으로 "
                     "확인한 결과 전부 매크로락톤/당 고리 내부의 고리형 에테르로, 우리 규칙이 "
                     "다루는 \"고리 밖 열린 사슬\" 상황과는 구조적으로 다름 - 이 계열은 직접적 "
                     "근거로 부적합함이 확인됨."},
    {"rule": "isolated_alkene", "type": "긍정_승인약물쌍",
     "description": "SIROLIMUS(시롤리무스), TACROLIMUS ANHYDROUS(타크로리무스) - 둘 다 ChEMBL "
                     "조회로 승인·비철수 확인됨(withdrawn_flag=False), 대형 매크로라이드 면역억제제로 "
                     "현재도 널리 처방됨. problem_smarts로 직접 매치되는 고립 지방족 알켄이 구조 "
                     "안에 실제 존재 - 고립 알켄이 항상 제거 대상은 아님을 보여주는 실제 승인약물 사례."},
    {"rule": "isolated_alkene", "type": "위험=메커니즘_참고_인과불명",
     "description": "CYCLOBARBITAL, HEXOBARBITAL 둘 다 ChEMBL 조회로 withdrawn_flag=True 확인됨, "
                     "둘 다 problem_smarts에 매치되는 사이클로헥세닐 고립 알켄 치환기를 가짐. "
                     "다만 바르비투르산염 계열은 호흡억제·의존성 등 일반적 안전성 문제로 철수된 "
                     "사례가 많아, 이 알켄 구조가 철수의 직접 원인이라는 인과관계는 확인되지 않음 "
                     "(상관관계만 관찰, 문헌 추가 확인 필요)."},
    {"rule": "nitro_group", "type": "위험=메커니즘_참고_검증완료",
     "description": "METRONIDAZOLE, NITROFURANTOIN, BENZNIDAZOLE 셋 다 ChEMBL 조회로 승인·비철수 "
                     "확인됨(max_phase=4.0, withdrawn_flag=False), SMILES에 니트로기([N+](=O)[O-]) "
                     "실제 존재 확인. 항균/항기생충제 계열에서 니트로기의 선택적 환원 활성화 자체가 "
                     "치료 메커니즘인 프로드러그 설계 사례 - 이런 계열에는 니트로기 제거가 "
                     "부적절함을 실제 조회로 검증함."},
]


def get_precedents(rule_name: str) -> str | None:
    """규칙 이름으로 관련 선례를 찾아 프롬프트에 넣을 텍스트로 반환."""
    matches = [p for p in PRECEDENT_LIBRARY if p['rule'] == rule_name]
    if not matches:
        return None
    return "\n".join([f"- [{m['type']}] {m['description']}" for m in matches])


Overwriting src/tools/precedent_library.py


In [28]:
importlib.reload(src.tools.precedent_library)
from src.tools.precedent_library import PRECEDENT_LIBRARY, get_precedents
clear_failure_memory()

print(f"총 선례 수: {len(PRECEDENT_LIBRARY)} (16이어야 정상)")
print(get_precedents('nitro_group'))

총 선례 수: 16 (16이어야 정상)
- [위험=메커니즘_참고_검증완료] METRONIDAZOLE, NITROFURANTOIN, BENZNIDAZOLE 셋 다 ChEMBL 조회로 승인·비철수 확인됨(max_phase=4.0, withdrawn_flag=False), SMILES에 니트로기([N+](=O)[O-]) 실제 존재 확인. 항균/항기생충제 계열에서 니트로기의 선택적 환원 활성화 자체가 치료 메커니즘인 프로드러그 설계 사례 - 이런 계열에는 니트로기 제거가 부적절함을 실제 조회로 검증함.


In [29]:
import ast

with open('src/tools/precedent_library.py') as f:
    source = f.read()
try:
    ast.parse(source)
    print("✅ precedent_library.py 문법 정상")
except SyntaxError as e:
    print("❌ 문법 오류:", e)

importlib.reload(src.tools.precedent_library)
from src.tools.precedent_library import PRECEDENT_LIBRARY, get_precedents

assert len(PRECEDENT_LIBRARY) == 16, f"개수 불일치: {len(PRECEDENT_LIBRARY)}"
print(f"✅ 총 {len(PRECEDENT_LIBRARY)}건")

required_keys = {'rule', 'type', 'description'}
for i, p in enumerate(PRECEDENT_LIBRARY):
    missing = required_keys - p.keys()
    assert not missing, f"{i}번째 항목에 키 누락: {missing}"
print("✅ 모든 항목 필수 키 정상")

for rule in ['Aliphatic_long_chain', 'isolated_alkene', 'nitro_group']:
    result = get_precedents(rule)
    assert result is not None, f"{rule} 조회 실패"
    print(f"✅ {rule}: {result.count(chr(10))+1}줄 반환")

importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory
clear_failure_memory()
smoke_result = iterative_fix_loop('CCCCCCCCCCCCCCCC', max_iterations=10, candidate_idx=0)
assert smoke_result['status'] == 'success', f"스모크 테스트 실패: {smoke_result['status']}"
print("✅ iterative_fix_loop 스모크 테스트 통과")

print("\n전체 통과 — 커밋해도 안전합니다.")

✅ precedent_library.py 문법 정상
✅ 총 16건
✅ 모든 항목 필수 키 정상
✅ Aliphatic_long_chain: 2줄 반환
✅ isolated_alkene: 2줄 반환
✅ nitro_group: 1줄 반환
✅ iterative_fix_loop 스모크 테스트 통과

전체 통과 — 커밋해도 안전합니다.


In [30]:
!cd /content/laidd-2026 && git add . && git commit -m "Add nitro_group precedent to PRECEDENT_LIBRARY (metronidazole/nitrofurantoin/benznidazole prodrug mechanism, verified)" && git push

[main 97129b1] Add nitro_group precedent to PRECEDENT_LIBRARY (metronidazole/nitrofurantoin/benznidazole prodrug mechanism, verified)
 1 file changed, 6 insertions(+)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 875 bytes | 875.00 KiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   50eeba7..97129b1  main -> main


In [31]:
info = get_replacement_candidates('aniline')
print("problem_smarts:", info['problem_smarts'])
for i, c in enumerate(info['candidates']):
    print(f"\ncandidate idx={i}")
    for k, v in c.items():
        print(f"  {k}: {v}")

problem_smarts: [NH2]c1ccc([#6,#7,#8,#16])cc1

candidate idx=0
  edit_type: add_substituent
  param: C(=O)C
  target_idx_in_pattern: 0
  name: acetamide (acylated amine)
  rationale: [참고] 설파계 항생제(설파닐아마이드, 설파메톡사졸 등)와 프로카인아마이드처럼 아닐린 골격이 반응성 대사가 아닌 안정적 형태로 널리 처방되어 온 사례가 다수 있음. 이 경우 특이체질 반응은 드물고 예측이 어려워, 본 경고를 절대적 배제 기준이 아닌 참고 신호로 해석해야 함. || 1차 방향족 아민을 아마이드로 아실화하여 N-hydroxylation 경로 자체를 차단

candidate idx=1
  edit_type: replace_ring
  param: [*:1]C12CC(C1)(C2)[*:2]
  ring_atom_indices_in_pattern: [1, 2, 3, 4, 6, 7]
  anchor_indices_in_pattern: (0, 5)
  name: BCP (bicyclo[1.1.1]pentane)
  rationale: para-이치환 아닐린의 방향족 벤젠 고리를 포화 bicyclic 탄소골격(BCP)으로 교체함. 방향족성 제거로 aniline reactive metabolite(RM) 형성 및 CYP-inhibition을 감소시켜, 퀴논이민 생성 경로를 차단하고 특이체질 약물 부작용(IADR) 위험을 낮춤 (문헌 근거, 학생 제공). 벤젠과의 공간적 유사성, Fsp3 증가, 실제 성공 사례가 많아 채택. 아마이드화(단순 아민 치환)보다 변화 폭이 크지만, 물성 개선 효과도 더 큼


In [32]:
for name in ['SULFANILAMIDE', 'SULFAMETHOXAZOLE', 'PROCAINAMIDE']:
    res = list(molecule.filter(pref_name__iexact=name))
    for r in res[:3]:
        print(r['molecule_chembl_id'], r.get('pref_name'), 'max_phase=', r.get('max_phase'), 'withdrawn=', r.get('withdrawn_flag'))
        smi = r.get('molecule_structures', {}).get('canonical_smiles') if r.get('molecule_structures') else None
        print('  smiles:', smi)

CHEMBL21 SULFANILAMIDE max_phase= 4.0 withdrawn= False
  smiles: Nc1ccc(S(N)(=O)=O)cc1
CHEMBL443 SULFAMETHOXAZOLE max_phase= 4.0 withdrawn= False
  smiles: Cc1cc(NS(=O)(=O)c2ccc(N)cc2)no1
CHEMBL640 PROCAINAMIDE max_phase= 4.0 withdrawn= False
  smiles: CCN(CC)CCNC(=O)c1ccc(N)cc1


In [33]:
%%writefile src/tools/precedent_library.py


"""선례 라이브러리 — 승인/철수 약물, 정량 활성 데이터, 도킹 검증 결과를
판단 에이전트 프롬프트에 실시간 주입하기 위한 구조화된 근거 저장소.
모든 항목은 이 세션에서 ChEMBL/GtoPdb API 조회 또는 실제 도킹 실행으로
직접 확인한 것만 포함한다(추정/일반 지식은 배제).
"""

PRECEDENT_LIBRARY = [
    {"rule": "Thiocarbonyl_group", "type": "긍정_승인약물쌍",
     "description": "티오펜탈(C=S)/펜토바비탈(C=O), 티아밀랄(C=S)/세코바비탈(C=O) - "
                     "동일 사이드체인, C=S->C=O만 다른 실제 승인 마취제 쌍. "
                     "baseline 모델 기준 옥소형이 티오형보다 Tox21 평균 예측값 낮음(-0.008~-0.009)."},
    {"rule": "catechol", "type": "정량_활성데이터",
     "description": "도파민이 D1(Ki 4.3-5.6nM)/D2(Ki 4.7-7.2nM)/D3(Ki 6.4-7.3nM) 수용체에 "
                     "단자릿수 nM 강력 결합 - 카테콜 골격이 활성에 필수적임을 정량적으로 뒷받침."},
    {"rule": "hydroxamic_acid", "type": "부정_참고사례_검증필요",
     "description": "하이드록삼산 골격(보리노스타트 등 HDAC 억제제)은 아연 킬레이션이 "
                     "약효 핵심이므로, 이 계열에 대한 무분별한 치환은 약효 상실 위험. "
                     "(문헌 재확인 필요)"},
    {"rule": "beta-keto/anhydride", "type": "긍정_통계검증결과",
     "description": "MMPDB 공식 통계 도구로 재검증한 결과, Tox21 규모(1173개)에서 "
                     "무수물 관련 매칭쌍은 표본 부족(count=1)으로 통계적 유의성 확보 불가 - "
                     "데이터형 접근보다 문헌형 근거가 더 신뢰할 만함을 시사."},
    {"rule": "Michael_acceptor_1", "type": "위험=메커니즘_참고",
     "description": "에타크린산(이뇨제, FDA 승인)은 시스테인 잔기와의 공유결합 자체가 "
                     "작용 메커니즘인 공유결합 억제제 - Michael acceptor 경고가 항상 "
                     "제거 대상은 아님을 보여주는 실제 승인약물 사례."},
    {"rule": "alkyl_halide", "type": "위험=메커니즘_참고",
     "description": "메클로르에타민, 사이클로포스파미드 등 알킬화 항암제는 DNA 알킬화 "
                     "반응성 자체가 세포독성 치료 메커니즘 - 이 계열에는 할로겐 제거가 "
                     "부적절함을 보여주는 실제 승인약물 사례."},
    {"rule": "azo_A(324)", "type": "위험=메커니즘_참고_검증완료",
     "description": "설파살라진(SMILES 내 /N=N/ 아조 결합 확인, ChEMBL max_phase=4.0, "
                     "GtoPdb FDA 승인 1950년/WHO 필수의약품)은 아조 결합이 장내 "
                     "세균에 의해 환원되어 활성 대사물(5-ASA)을 방출하는 프로드러그 - "
                     "실제 조회로 검증됨."},
    {"rule": "catechol", "type": "도킹검증_결과",
     "description": "COMT(PDB 1VID) 도킹 검증: 도파민(-5.72 kcal/mol)→메톡시도파민"
                     "(-5.41 kcal/mol), 변화폭 +0.31 kcal/mol로 약화 방향이나 이는 "
                     "1 kcal/mol 미만의 작은 차이로 도킹 자체의 오차범위 내일 수 있어 "
                     "단정적 근거로 삼기엔 약함. 에피네프린은 반대로 미세 강화"
                     "(-6.21→-6.32, -0.10) - 두 경우 모두 변화폭이 작아, 도킹 수치보다는 "
                     "카테콜의 수용체 결합 필수성(정성적 근거)이 더 강한 판단 기준."},
    {"rule": "Michael_acceptor_1", "type": "도킹검증_방법론한계",
     "description": "EGFR(PDB 6JX4) 도킹 검증: 오시메르티닙(-7.13)→C=C환원버전(-7.08), "
                     "거의 무변화(+0.05). 표준(비공유) 도킹이 오시메르티닙의 실제 "
                     "공유결합(Cys797) 메커니즘을 포착하지 못하는 방법론적 한계 확인 - "
                     "공유결합 억제제 계열은 일반 도킹 스코어만으로 활성 손실을 판단하지 "
                     "말 것(도킹 무변화가 곧 활성 유지를 뜻하지 않음)."},
    {"rule": "hydroquinone", "type": "도킹검증_결과",
     "description": "NQO1 도킹 검증: 퀴논(-3.29)→하이드로퀴논(-4.08), 결합 강화(-0.79, "
                     "1 kcal/mol에 근접하는 뚜렷한 변화). 메틸퀴논(-3.80)→환원버전"
                     "(-4.29)도 강화(-0.49), 2건 모두 일관되게 강화 방향. NQO1이 실제로 "
                     "퀴논을 하이드로퀴논으로 환원하는 효소이므로, 이 치환 방향은 해독 "
                     "반응경로와 자연스럽게 정렬되며 실측 결합력도 개선됨 - 활성 손실 "
                     "우려가 낮은 것으로 확인됨."},
    {"rule": "quinone_A(370)", "type": "도킹검증_결과",
     "description": "NQO1 도킹 검증: 퀴논(-3.29)→하이드로퀴논(-4.08), 결합 강화(-0.79). "
                     "실제 표적 효소와의 결합력이 오히려 개선되는 것으로 실측 확인됨 "
                     "(hydroquinone 규칙과 동일 표적 데이터 공유)."},
    {"rule": "Aliphatic_long_chain", "type": "긍정_승인약물_확인(구조는_동의어로_대체확인)",
     "description": "POLIDOCANOL(라우릴알코올+에틸렌옥사이드 평균 9개 반복부가체)은 ChEMBL 조회로 "
                     "승인 확인됨(max_phase=4.0, first_approval=2010, ATC C05BB02, "
                     "dosed_ingredient=True, withdrawn=False, 상품명 Asclera/Aethoxysklerol). "
                     "ChEMBL에 단일 SMILES는 없으나(polymer_flag=1, structure_type=NONE) "
                     "이는 다분산 고분자라 원천적으로 단일 구조가 없기 때문이며, 공식 동의어"
                     "(USP: Polyoxyl 9 lauryl ether, JAN: Lauromacrogol 400)가 "
                     "\"장쇄 알킬+반복 에테르\" 구조를 명확히 정의함 - 실제 승인약물에서 "
                     "이 전략이 쓰이고 있음을 뒷받침."},
    {"rule": "Aliphatic_long_chain", "type": "부정_참고사례_검증필요",
     "description": "ChEMBL 서브구조 검색(에테르 삽입 사슬 모티프)으로 매치된 승인약물은 "
                     "에리스로마이신/아지스로마이신/암포테리신B였으나, 매치 위치를 IsInRing으로 "
                     "확인한 결과 전부 매크로락톤/당 고리 내부의 고리형 에테르로, 우리 규칙이 "
                     "다루는 \"고리 밖 열린 사슬\" 상황과는 구조적으로 다름 - 이 계열은 직접적 "
                     "근거로 부적합함이 확인됨."},
    {"rule": "isolated_alkene", "type": "긍정_승인약물쌍",
     "description": "SIROLIMUS(시롤리무스), TACROLIMUS ANHYDROUS(타크로리무스) - 둘 다 ChEMBL "
                     "조회로 승인·비철수 확인됨(withdrawn_flag=False), 대형 매크로라이드 면역억제제로 "
                     "현재도 널리 처방됨. problem_smarts로 직접 매치되는 고립 지방족 알켄이 구조 "
                     "안에 실제 존재 - 고립 알켄이 항상 제거 대상은 아님을 보여주는 실제 승인약물 사례."},
    {"rule": "isolated_alkene", "type": "위험=메커니즘_참고_인과불명",
     "description": "CYCLOBARBITAL, HEXOBARBITAL 둘 다 ChEMBL 조회로 withdrawn_flag=True 확인됨, "
                     "둘 다 problem_smarts에 매치되는 사이클로헥세닐 고립 알켄 치환기를 가짐. "
                     "다만 바르비투르산염 계열은 호흡억제·의존성 등 일반적 안전성 문제로 철수된 "
                     "사례가 많아, 이 알켄 구조가 철수의 직접 원인이라는 인과관계는 확인되지 않음 "
                     "(상관관계만 관찰, 문헌 추가 확인 필요)."},
    {"rule": "nitro_group", "type": "위험=메커니즘_참고_검증완료",
     "description": "METRONIDAZOLE, NITROFURANTOIN, BENZNIDAZOLE 셋 다 ChEMBL 조회로 승인·비철수 "
                     "확인됨(max_phase=4.0, withdrawn_flag=False), SMILES에 니트로기([N+](=O)[O-]) "
                     "실제 존재 확인. 항균/항기생충제 계열에서 니트로기의 선택적 환원 활성화 자체가 "
                     "치료 메커니즘인 프로드러그 설계 사례 - 이런 계열에는 니트로기 제거가 "
                     "부적절함을 실제 조회로 검증함."},
    {"rule": "aniline", "type": "위험=메커니즘_참고_검증완료",
     "description": "SULFANILAMIDE, SULFAMETHOXAZOLE, PROCAINAMIDE 셋 다 ChEMBL 조회로 승인·비철수 "
                     "확인됨(max_phase=4.0, withdrawn_flag=False), SMILES 확인 결과 셋 다 아실화되지 "
                     "않은 유리 1차 방향족 아민(아닐린) 형태로 실제 처방됨. 설파계 항생제·항부정맥제 "
                     "계열에서 특이체질 반응 위험에도 불구하고 유리 아닐린 골격이 오랜 기간 널리 "
                     "쓰여온 사례 - 이 경고가 절대적 배제 기준이 아님을 실제 조회로 검증함."},
]


def get_precedents(rule_name: str) -> str | None:
    """규칙 이름으로 관련 선례를 찾아 프롬프트에 넣을 텍스트로 반환."""
    matches = [p for p in PRECEDENT_LIBRARY if p['rule'] == rule_name]
    if not matches:
        return None
    return "\n".join([f"- [{m['type']}] {m['description']}" for m in matches])


Overwriting src/tools/precedent_library.py


In [34]:
import ast

with open('src/tools/precedent_library.py') as f:
    source = f.read()
ast.parse(source)
print("✅ 문법 정상")

importlib.reload(src.tools.precedent_library)
from src.tools.precedent_library import PRECEDENT_LIBRARY, get_precedents

assert len(PRECEDENT_LIBRARY) == 17, f"개수 불일치: {len(PRECEDENT_LIBRARY)}"
print(f"✅ 총 {len(PRECEDENT_LIBRARY)}건")

required_keys = {'rule', 'type', 'description'}
for i, p in enumerate(PRECEDENT_LIBRARY):
    assert not (required_keys - p.keys()), f"{i}번째 항목 키 누락"
print("✅ 모든 항목 필수 키 정상")

print(get_precedents('aniline'))

importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory
clear_failure_memory()
smoke_result = iterative_fix_loop('CCCCCCCCCCCCCCCC', max_iterations=10, candidate_idx=0)
assert smoke_result['status'] == 'success'
print("✅ 스모크 테스트 통과")
print("\n전체 통과 — 커밋해도 안전합니다.")

✅ 문법 정상
✅ 총 17건
✅ 모든 항목 필수 키 정상
- [위험=메커니즘_참고_검증완료] SULFANILAMIDE, SULFAMETHOXAZOLE, PROCAINAMIDE 셋 다 ChEMBL 조회로 승인·비철수 확인됨(max_phase=4.0, withdrawn_flag=False), SMILES 확인 결과 셋 다 아실화되지 않은 유리 1차 방향족 아민(아닐린) 형태로 실제 처방됨. 설파계 항생제·항부정맥제 계열에서 특이체질 반응 위험에도 불구하고 유리 아닐린 골격이 오랜 기간 널리 쓰여온 사례 - 이 경고가 절대적 배제 기준이 아님을 실제 조회로 검증함.
✅ 스모크 테스트 통과

전체 통과 — 커밋해도 안전합니다.


In [35]:
!cd /content/laidd-2026 && git add . && git commit -m "Add aniline precedent to PRECEDENT_LIBRARY (sulfanilamide/sulfamethoxazole/procainamide free-aniline drugs, verified)" && git push

[main ae41df2] Add aniline precedent to PRECEDENT_LIBRARY (sulfanilamide/sulfamethoxazole/procainamide free-aniline drugs, verified)
 1 file changed, 6 insertions(+)
Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 2 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 865 bytes | 865.00 KiB/s, done.
Total 5 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 3 local objects.
To https://github.com/Dec32th/laidd-2026.git
   97129b1..ae41df2  main -> main


In [36]:
!cat src/tools/agent.py

import json
from src.tools.replacement_library import get_replacement_candidates


def _call_llm(client, model_name, prompt, client_type="gemini"):
    """client_type에 따라 Gemini SDK 또는 OpenAI 호환 SDK로 호출하고,
    응답 텍스트만 통일된 형태로 반환."""
    if client_type == "gemini":
        response = client.models.generate_content(model=model_name, contents=prompt)
        return response.text
    elif client_type == "openai_compatible":
        response = client.chat.completions.create(
            model=model_name,
            messages=[{"role": "user", "content": prompt}]
        )
        return response.choices[0].message.content
    else:
        raise ValueError(f"알 수 없는 client_type: {client_type}")


def _parse_json_response(text, fallback):
    text = text.strip()
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'):
            text = text[4:]
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return fallback


def

In [43]:
from openai import OpenAI
from google.colab import userdata

dashscope_key = userdata.get('DASHSCOPE_API_KEY')
client_qwen = OpenAI(api_key=dashscope_key, base_url="https://token-plan.ap-southeast-1.maas.aliyuncs.com/compatible-mode/v1")

In [44]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
importlib.reload(src.tools.agent)
importlib.reload(src.tools.precedent_library)  # 추가 필요
from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory
clear_failure_memory()

import random
random.seed(7)
sample_smiles = random.sample(list(data['smiles_valid']), 50)

from collections import Counter
llm_status_counter = Counter()
llm_results = []

for smi in sample_smiles:
    r = iterative_fix_loop(
        smi, max_iterations=10, candidate_idx=0,
        llm_client=client_qwen, llm_model="qwen3.8-max", llm_client_type="openai_compatible",
    )
    llm_status_counter[r['status']] += 1
    llm_results.append((smi, r))

print("--- LLM(Qwen) 기반 판단 결과 (샘플 50개) ---")
for status, count in llm_status_counter.most_common():
    print(f"{status}: {count}")

review_cases = [(smi, r) for smi, r in llm_results
                 for detail in r.get('skipped_details', [])
                 if '보류' in detail.get('reason', '')]
print(f"\nLLM이 치환 보류(사람 검토 권장) 판단한 케이스: {len(review_cases)}건")
for smi, r in review_cases[:5]:
    print(' ', smi)
    for detail in r.get('skipped_details', []):
        if '보류' in detail.get('reason', ''):
            print('   reason:', detail['reason'])

--- LLM(Qwen) 기반 판단 결과 (샘플 50개) ---
success: 33
no_known_fix: 9
stuck: 8

LLM이 치환 보류(사람 검토 권장) 판단한 케이스: 2건
  O=C1OC(CN2CCOCC2)CN1N=Cc1ccc([N+](=O)[O-])o1
   reason: LLM이 치환을 보류했습니다: 이 분자는 니트로푸란토인과 유사한 니트로푸릴 히드라존 계열로 보여 니트로기의 환원 활성화가 약효에 필요할 가능성이 있으므로 치환을 보류하고 사람 검토가 필요합니다. (이 분자가 [참고] 사항에 해당하는 안전한 실사용 사례와 유사하다고 판단되어, 자동 치환 대신 연구자의 직접 검토를 권장합니다.)
  CC(C)CN(C[C@@H](OP(=O)([O-])[O-])[C@H](Cc1ccccc1)NC(=O)O[C@H]1CCOC1)S(=O)(=O)c1ccc(N)cc1
   reason: LLM이 치환을 보류했습니다: 이 분자는 승인된 설폰아마이드 계열 약물처럼 p-아미노페닐설폰아마이드 골격을 포함하는 참고 사례 유사체이므로, 자동 치환보다 사람 검토가 적절하다. (이 분자가 [참고] 사항에 해당하는 안전한 실사용 사례와 유사하다고 판단되어, 자동 치환 대신 연구자의 직접 검토를 권장합니다.)


In [45]:
importlib.reload(src.tools.replacement_library)
importlib.reload(src.tools.atom_editor)
importlib.reload(src.tools.molecule_editor)
from src.tools.molecule_editor import iterative_fix_loop, clear_failure_memory
clear_failure_memory()

rule_status_counter = Counter()
rule_results = {}

for smi in sample_smiles:
    r = iterative_fix_loop(smi, max_iterations=10, candidate_idx=0)
    rule_status_counter[r['status']] += 1
    rule_results[smi] = r['status']

print("--- 규칙 기반(LLM 없음) 판단 결과 (동일 50개 샘플) ---")
for status, count in rule_status_counter.most_common():
    print(f"{status}: {count}")

# 분자별로 status가 달라진 케이스만 비교
llm_results_dict = {smi: r['status'] for smi, r in llm_results}

print("\n--- 규칙 기반 vs LLM(Qwen) 결과가 달라진 분자 ---")
diff_count = 0
for smi in sample_smiles:
    rb = rule_results[smi]
    llm = llm_results_dict[smi]
    if rb != llm:
        diff_count += 1
        print(f"[{rb} -> {llm}] {smi}")

print(f"\n총 {diff_count}/50건에서 판단이 달라짐")

--- 규칙 기반(LLM 없음) 판단 결과 (동일 50개 샘플) ---
success: 34
stuck: 8
no_known_fix: 8

--- 규칙 기반 vs LLM(Qwen) 결과가 달라진 분자 ---
[success -> no_known_fix] O=C1OC(CN2CCOCC2)CN1N=Cc1ccc([N+](=O)[O-])o1

총 1/50건에서 판단이 달라짐
